Uninstalling torchao to resolve compatibility conflict

In [1]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


Magnitude Pruning Step 3: Masked Merge of LoRA Adapter into FP16 CodeLlama

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from peft.tuners.lora import Linear as LoraLinear

model_name = "codellama/CodeLlama-7b-Instruct-hf"   # tokenizer source (pruning never changes it)
base_dir = "/content/pruned_base"
adapter_dir = "/content/pruned_adapter"
save_dir = "/content/pruned_model"

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU found. Set the Colab runtime to T4.")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(base_dir, dtype=torch.float16, device_map="cuda")
model = PeftModel.from_pretrained(model, adapter_dir)

# --- Masked merge: W += (B @ A) * scaling, applied only where W != 0.
# Done one layer at a time, so no global mask is ever held in memory.
with torch.no_grad():
    for name, module in model.named_modules():
        if isinstance(module, LoraLinear):
            W = module.base_layer.weight
            A = module.lora_A["default"].weight.float()
            B = module.lora_B["default"].weight.float()
            delta = (B @ A) * module.scaling["default"]
            W += (delta * (W != 0)).to(W.dtype)
            del A, B, delta
model = model.unload()            # drop the LoRA wrappers; base weights already hold the update
torch.cuda.empty_cache()

# --- Re-check sparsity: must match step 1 (40.08% in pruned layers)
pruned_total, zero_total = 0, 0
for layer in model.model.layers:
    for module in layer.modules():
        if isinstance(module, torch.nn.Linear):
            pruned_total += module.weight.numel()
            zero_total += (module.weight == 0).sum().item()
print(f"Sparsity after merge: {zero_total / pruned_total:.2%} "
      f"({zero_total:,} of {pruned_total:,} weights)")

# --- "After" coherence check: same greedy generation as step 1
Code = '''from typing import List
def below_zero(operations: List[int]) -> bool:
    """ You're given a list of deposit and withdrawal operations
    on a bank account that starts with zero balance. Your task is to
    detect if at any point the balance of account falls below zero,
    and at that point function should return True.
    """
    balance = 0
    for op in operations:
        balance += op
        if balance < 0:
            return True
    return False'''

prompt = ("Generate Google-style docstring for the following Python function: {code}. "
          "Only output the docstring and function code.").format(code=Code)   # P-2

inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}], add_generation_prompt=True,
    return_tensors="pt", return_dict=True
).to(model.device)

model.eval()
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=300, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
n_new = out.shape[1] - inputs["input_ids"].shape[1]
print(f"\n--- Coherence check after fine-tune ({n_new} tokens, limit 300) ---")
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

# --- Save the final pruned model (FP16, dense format). Wait for the last line before restarting.
# Small shards: each shard is staged in system RAM while it is written (Colab RAM ~12.7 GB).
model.save_pretrained(save_dir, max_shard_size="2GB")
print(f"\nFinal pruned model saved to {save_dir}")


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Sparsity after merge: 40.08% (2,595,783,122 of 6,476,005,376 weights)

--- Coherence check after fine-tune (115 tokens, limit 300) ---
```
def below_zero(operations: List[int]) -> bool:
    """ You're given a list of deposit and withdrawal operations
    on a bank account that starts with zero balance. Your task is to
    detect if at any point the balance of account falls below zero,
    and at that point function should return True.
    """
    balance = 0
    for op in operations:
        balance += op
        if balance < 0:
            return True
    return False
```


Writing model shards:   0%|          | 0/7 [00:00<?, ?it/s]


Final pruned model saved to /content/pruned_model
